In [3]:
!pip install -U transformers datasets accelerate bitsandbytes peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  #Enable 4-bit quantization for model loading
    bnb_4bit_quant_type="nf4", #nf4 quantization format for better performance and accuracy
    bnb_4bit_compute_dtype=torch.float16, #Use float16 for computations to balance performance and precision
    bnb_4bit_use_double_quant=True, #Enable double quantization to further reduce memory usage while maintaining accuracy
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False

print("Model loaded successfully in 4-bit mode")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded successfully in 4-bit mode


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,#ranking of the low-rank decomposition
    lora_alpha=32, #scaling factor for the low-rank decomposition
    lora_dropout=0.05, #dropout rate for the LoRA layers prevent overfitting
    target_modules=[
        "q_proj",
        "v_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/drive/MyDrive/data/train.jsonl",
        "validation": "/content/drive/MyDrive/data/val.jsonl",
    },
)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 3969
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 441
    })
})

Sample training example:
{'instruction': 'Answer this HR question about an employee', 'input': 'Employee #1465: Age 45, Male, Manufacturing Director, Dept: Research & Development, Salary: $9,380, Satisfaction: 4/5, Attrition: No', 'output': 'Profile: Male Manufacturing Director (age 45) in Research & Development earns $9,380/month. Satisfaction: 4/5. Stayed. 10yr exp.'}


In [ ]:
def format_prompt(example):
    instruction = example["instruction"]
    input_text = example["input"]
    output = example["output"]

    if input_text.strip() != "":
        text = (
            "<|user|>\n"
            f"{instruction}\n\n"
            "Input:\n"
            f"{input_text}\n"
            "<|assistant|>\n"
            f"{output}"
        )
    else:
        text = (
            "<|user|>\n"
            f"{instruction}\n"
            "<|assistant|>\n"
            f"{output}"
        )

    return {"text": text}

dataset = dataset.map(
    format_prompt,
    remove_columns=dataset["train"].column_names,
)

print(dataset)
print("\nFormatted sample:\n")
print(dataset["train"][0]["text"])

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 3969
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 441
    })
})

Formatted sample:

<|user|>
Answer this HR question about an employee

Input:
Employee #1465: Age 45, Male, Manufacturing Director, Dept: Research & Development, Salary: $9,380, Satisfaction: 4/5, Attrition: No
<|assistant|>
Profile: Male Manufacturing Director (age 45) in Research & Development earns $9,380/month. Satisfaction: 4/5. Stayed. 10yr exp.


In [ ]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="./outputs",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4, #LEARNING rate is a step size for updating model parameters during training.
)

In [13]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=sft_config,
)

Adding EOS to train dataset:   0%|          | 0/3969 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3969 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3969 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/441 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/441 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/441 [00:00<?, ? examples/s]

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
trainer.train()
trainer.model.save_pretrained("/content/drive/MyDrive/adapters")
tokenizer.save_pretrained("/content/drive/MyDrive/adapters")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
200,0.186365,0.186467
400,0.177380,0.181763
600,0.178429,0.178411


Step,Training Loss,Validation Loss
200,0.186365,0.186467
400,0.177380,0.181763
600,0.178429,0.178411
800,0.181218,0.180789
1000,0.182177,0.183629
1200,0.180729,0.177694
1400,0.175950,0.177182
1600,0.176562,0.176640
1800,0.168097,0.176911
2000,0.172852,0.176297


('/content/drive/MyDrive/adapters/tokenizer_config.json',
 '/content/drive/MyDrive/adapters/chat_template.jinja',
 '/content/drive/MyDrive/adapters/tokenizer.json')

In [15]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "/content/drive/MyDrive/adapters"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype="auto"
)

model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [16]:
prompt = """<|user|>
Analyze this employee profile:

Employee #512, Age 38, Role: HR Manager, Dept: HR, Salary: $10,200, Satisfaction: 4/5
<|assistant|>
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.7,
    do_sample=True
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

<|user|>
Analyze this employee profile:

Employee #512, Age 38, Role: HR Manager, Dept: HR, Salary: $10,200, Satisfaction: 4/5
<|assistant|>
Profile: Employee #512 is a HR Manager in the HR department, earning $10,200/month. Satisfaction: 4/5. 10yr exp.
